# ToM V6 Warm-Start 140k Review (Apr 14 Only)

This notebook is a clean, single-purpose Apr 14 analysis.

Scope:
- Compare warm-start baselines from `modal/tom-experiment-incumbent/v6-omx-full2-promo-candidate`
- Against 140k continuations in `modal/tom-140k-modal-results-v6`
- Seeds: `7`, `11`, `23`
- Report per-seed deltas, aggregate means, and quick-gate style verdict


In [ ]:
import json
from pathlib import Path

try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown:
        def __init__(self, text):
            self.text = text
        def __repr__(self):
            return self.text
    def display(*objs):
        for obj in objs:
            print(obj)

try:
    import pandas as pd
except ImportError:
    pd = None


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for base in [start, *start.parents]:
        if all((base / name).exists() for name in ('train.py', 'env.py', 'eval.py')):
            return base
    fallback = Path('/Users/stephenbeale/Projects/ToM_AI_Research_Team')
    if all((fallback / name).exists() for name in ('train.py', 'env.py', 'eval.py')):
        return fallback
    raise FileNotFoundError('Could not locate benchmark repo root')


def load_json(path: Path):
    return json.loads(path.read_text())


def as_table(rows, index=None, sort_by=None, ascending=True):
    if pd is None:
        return rows
    df = pd.DataFrame(rows)
    if sort_by is not None and len(df):
        df = df.sort_values(sort_by, ascending=ascending)
    if index is not None and len(df):
        df = df.set_index(index)
    return df


ROOT = find_repo_root()
display(Markdown(f"**Repo root:** `{ROOT}`"))


In [ ]:
V6_INCUMBENT_ROOT = ROOT / 'modal' / 'tom-experiment-incumbent' / 'v6-omx-full2-promo-candidate'
V6_RESULTS_ROOT = ROOT / 'modal' / 'tom-140k-modal-results-v6'
SEEDS = [7, 11, 23]

v6_incumbent = {
    seed: load_json(V6_INCUMBENT_ROOT / f'seed{seed}' / 'run_summary.json')
    for seed in SEEDS
}
v6_results = {
    seed: load_json(V6_RESULTS_ROOT / f'seed{seed}' / 'target-140000' / 'run_summary.json')
    for seed in SEEDS
}
v6_status = {
    seed: load_json(V6_RESULTS_ROOT / f'seed{seed}' / 'target-140000' / 'run_status.json')
    for seed in SEEDS
}

artifact_rows = []
for seed in SEEDS:
    artifact_rows.append({
        'seed': seed,
        'incumbent_run_summary': str(V6_INCUMBENT_ROOT / f'seed{seed}' / 'run_summary.json'),
        'result_run_summary': str(V6_RESULTS_ROOT / f'seed{seed}' / 'target-140000' / 'run_summary.json'),
        'result_run_status': str(V6_RESULTS_ROOT / f'seed{seed}' / 'target-140000' / 'run_status.json'),
        'status_state': v6_status[seed].get('state'),
        'completed_total_episodes': v6_status[seed].get('completed_total_episodes'),
    })

display(Markdown('## Artifact and Completion Status'))
display(as_table(artifact_rows, sort_by=['seed'], ascending=True))


In [ ]:
METRICS = [
    'ToMCoordScore',
    'DeadlockRate',
    'CollisionRate',
    'SuccessRate',
    'AverageDelay',
    'IntentionPredictionF1',
    'AmbiguityEfficiency',
    'CoordinationEfficiency',
    'StrategySwitchAccuracy',
]

delta_rows = []
for seed in SEEDS:
    baseline = v6_incumbent[seed]['selection']['candidate_metrics']
    candidate = v6_results[seed]['eval_metrics']
    row = {'seed': seed}
    for metric in METRICS:
        row[f'baseline_{metric}'] = baseline[metric]
        row[f'candidate_{metric}'] = candidate[metric]
        row[f'delta_{metric}'] = candidate[metric] - baseline[metric]
    delta_rows.append(row)

mean_row = {'seed': 'mean'}
for metric in METRICS:
    mean_row[f'baseline_{metric}'] = sum(r[f'baseline_{metric}'] for r in delta_rows) / len(delta_rows)
    mean_row[f'candidate_{metric}'] = sum(r[f'candidate_{metric}'] for r in delta_rows) / len(delta_rows)
    mean_row[f'delta_{metric}'] = sum(r[f'delta_{metric}'] for r in delta_rows) / len(delta_rows)

display(Markdown('## Per-Seed and Mean Deltas (`140k - incumbent`)'))
if pd is None:
    display(delta_rows + [mean_row])
else:
    display(pd.DataFrame(delta_rows + [mean_row]).set_index('seed'))



In [ ]:
mean_deadlock_delta = mean_row['delta_DeadlockRate']
mean_tom_delta = mean_row['delta_ToMCoordScore']
max_seed_deadlock_delta = max(r['delta_DeadlockRate'] for r in delta_rows)

quick_gate_pass = (
    mean_deadlock_delta <= 0
    and mean_tom_delta > 0
    and max_seed_deadlock_delta <= 0.10
)

summary_lines = [
    '## Quick-Gate Style Verdict',
    f"- Mean `ToMCoordScore` delta: `{mean_tom_delta:+.6f}`",
    f"- Mean `DeadlockRate` delta: `{mean_deadlock_delta:+.6f}`",
    f"- Max per-seed `DeadlockRate` delta: `{max_seed_deadlock_delta:+.6f}` (cap `+0.10`)",
    f"- Outcome: `{'PASS' if quick_gate_pass else 'FAIL'}`",
]

display(Markdown('\n'.join(summary_lines)))

